In [1]:
# Import necessary libraries
import os
from pathlib import Path
import numpy as np
import pandas as pd
from astropy import coordinates as coords
from astropy.coordinates import SkyCoord
from astropy import units as u
from astropy.table import Table
from astropy.io import fits
from astropy.cosmology import LambdaCDM
from tqdm import tqdm
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# Set cosmology
cosmo = LambdaCDM(H0=70, Om0=0.3, Ode0=0.7)

plt.rcParams.update({
    "font.family": 'STIXGeneral',
    'text.usetex': False,
    "mathtext.fontset": 'cm',
    "axes.labelweight": "bold",
    'font.size': 25,
    'font.weight': 'normal',
    
    # Tick direction and appearance
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.top': True,            # show top ticks
    'ytick.right': True,          # show right ticks
    'xtick.minor.visible': True,  # show minor x ticks
    'ytick.minor.visible': True,  # show minor y ticks
    'xtick.major.size': 10,
    'xtick.minor.size': 6,
    'ytick.major.size': 10,
    'ytick.minor.size': 6,
    'xtick.major.width': 1.6,
    'xtick.minor.width': 1.6,
    'ytick.major.width': 1.6,
    'ytick.minor.width': 1.6,
    
    # Axes and line properties
    'lines.linewidth': 2,
    'axes.linewidth': 3.5,
    'axes.labelpad': 4,
    'xtick.major.pad': 7,
    'image.origin': 'lower'
})
# Pandas configuration
pd.set_option('display.max_columns', None)

In [2]:
hecs_vac_data = pd.read_csv('../../DATA/AllHeCS_VAC_updated.csv')
Z_CLID = hecs_vac_data[hecs_vac_data['CLID']=='A2199']['Z'].values[0]
df0 = pd.read_csv('./A2199_mastercat_intermediate_file1.csv')

In [3]:
df0['p_modelmag_u_0'] = df0['p_modelmag_u'] - df0['p_extinction_u']
df0['p_modelmag_g_0'] = df0['p_modelmag_g'] - df0['p_extinction_g']
df0['p_modelmag_r_0'] = df0['p_modelmag_r'] - df0['p_extinction_r']
df0['p_modelmag_i_0'] = df0['p_modelmag_i'] - df0['p_extinction_i']
df0['p_modelmag_z_0'] = df0['p_modelmag_z'] - df0['p_extinction_z']

df0['p_petromag_u_0'] = df0['p_petromag_u'] - df0['p_extinction_u']
df0['p_petromag_g_0'] = df0['p_petromag_g'] - df0['p_extinction_g']
df0['p_petromag_r_0'] = df0['p_petromag_r'] - df0['p_extinction_r']
df0['p_petromag_i_0'] = df0['p_petromag_i'] - df0['p_extinction_i']  
df0['p_petromag_z_0'] = df0['p_petromag_z'] - df0['p_extinction_z']  

In [4]:
df0['grmod'] = df0['p_modelmag_g_0'] - df0['p_modelmag_r_0']

In [7]:
point_withz = df0[(df0['z_tot_z']!=-9) & (df0['p_probpsf']==1)]

# Make SDSS imglist-related files

In [ ]:
# Open a text file and write the header + data manually
with open('05b_point_source_withz_SDSSimglist_input.txt', 'w') as f:
    # f.write("index TARGET_RA TARGET_DEC\n")
    temp = point_withz.sort_values(by='p_modelmag_r', ascending=True)
    for idx, row in temp.iterrows():
        f.write(f"{row['p_objid']} {row['p_ra']:.6f} {row['p_dec']:.6f}\n")


In [ ]:
# Open a text file and write the header + data manually
with open('05c_point_source_withz_SDSSimglist_result.txt', 'w') as f:
    f.write("objid RA DEC SourceFlag\n")
    f.write("SourceFlag 0 = Invalid Source 1 = Point Source 2 = Extended Source\n")
    temp = point_withz.sort_values(by='p_modelmag_r', ascending=True)
    
    for idx, row in temp.iterrows():
        f.write(f"{row['p_objid']},{row['p_ra']:.6f},{row['p_dec']:.6f}, 1\n")

In [ ]:
import pandas as pd

file_path = "./05c_point_source_withz_SDSSimglist_result.txt"

vis = pd.read_csv(
    file_path,
    sep=r"\s*,\s*",
    engine="python",
    skiprows=2,
    names=["objid", "RA", "DEC", "SourceFlag"],
)

vis["SourceFlag"] = vis["SourceFlag"].astype(int)
vis.to_csv("./05d_point_source_withz_SDSSimglist_result.csv", index=False)